In [ ]:
# ============================================
# Blockwise permutation + Areal GP + VI module
# ============================================

import os
from typing import Dict, Any, Sequence

import numpy as np
import torch
import torch.optim as optim
from tqdm import tqdm

import pandas as pd
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()
except ImportError:
    ro = None  # meuse demo will be skipped if rpy2 not installed

from GPModel import GPModel
from GPArealModel import GPArealModel
from VIGP_Unlinked import VIGP_Unlinked

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _to_tensor(x, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)


# ---------------------------------------------------------
# STEP 2 helper: construct blockwise permutation at data level
# ---------------------------------------------------------

def make_blockwise_permuted_data(
    coords,
    X,
    Y,
    n_blocks: int,
    n_locations: int,
    seed: int = 521,
) -> Dict[str, torch.Tensor]:
    """
    Prepare original (restricted) and blockwise-permuted data.

    Inputs
    ------
    coords : array-like (N, d)
    X      : array-like (N,) or (N, 1)
    Y      : array-like (N,)
    n_blocks, n_locations : define N_use = n_blocks * n_locations
    seed : for reproducible permutations

    Returns a dict with:
        coords_orig, X_orig, Y_orig   : restricted, sorted (unpermuted)
        coords_perm, X_perm, Y_perm   : permuted inside blocks
        region_assignments            : (N_use,) with values 0,...,B-1
        perm_matrix_x, perm_matrix_s  : (N_use, N_use) block-diagonal perms
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    coords = _to_tensor(coords)
    Y = _to_tensor(Y).view(-1)  # (N,)
    X = _to_tensor(X)

    if X.ndim == 1:
        X = X.unsqueeze(1)  # (N, 1)

    N_total = coords.shape[0]
    N_use = n_blocks * n_locations
    if N_total < N_use:
        raise ValueError(f"Not enough locations: N={N_total}, required N_use={N_use}.")

    # Sort by first coordinate, then restrict to N_use
    sort_idx = torch.argsort(coords[:, 0])
    sort_idx = sort_idx[:N_use]

    coords_orig = coords[sort_idx].contiguous()  # (N_use, d)
    X_orig = X[sort_idx].contiguous()            # (N_use, 1)
    Y_orig = Y[sort_idx].contiguous()            # (N_use,)

    N = N_use

    # Region assignments: fixed blocks
    region_assignments = torch.zeros(N, dtype=torch.long, device=device)
    for b in range(n_blocks):
        start = b * n_locations
        end = (b + 1) * n_locations
        region_assignments[start:end] = b

    # Blockwise permutation matrices
    #   - perm_matrix_x: permutes X,Y jointly within block
    #   - perm_matrix_s: independent perm for coordinates
    perm_matrix_x = torch.zeros(N, N, device=device)
    perm_matrix_s = torch.zeros(N, N, device=device)

    for b in range(n_blocks):
        start = b * n_locations
        end = (b + 1) * n_locations

        # X,Y permutation
        perm_xy = torch.randperm(n_locations, device=device)
        for i in range(n_locations):
            perm_matrix_x[start + i, start + perm_xy[i]] = 1.0

        # S permutation
        perm_s = torch.randperm(n_locations, device=device)
        for i in range(n_locations):
            perm_matrix_s[start + i, start + perm_s[i]] = 1.0

    # Apply permutations
    Y_perm = (perm_matrix_x @ Y_orig.view(N, 1)).view(N)
    X_perm = perm_matrix_x @ X_orig                  # (N, 1)
    coords_perm = perm_matrix_s @ coords_orig        # (N, d)

    return {
        "coords_orig": coords_orig,
        "X_orig": X_orig,
        "Y_orig": Y_orig,
        "coords_perm": coords_perm,
        "X_perm": X_perm,
        "Y_perm": Y_perm,
        "region_assignments": region_assignments,
        "perm_matrix_x": perm_matrix_x,
        "perm_matrix_s": perm_matrix_s,
    }


# ---------------------------------------------------------
# STEP 3 helper: run Areal GP + VI on permuted data
# ---------------------------------------------------------

def run_areal_and_vi(
    coords_perm: torch.Tensor,
    X_perm: torch.Tensor,
    Y_perm: torch.Tensor,
    region_assignments: torch.Tensor,
    perm_matrix_x: torch.Tensor,
    perm_matrix_s: torch.Tensor,
    n_blocks: int,
    n_locations: int,
    seed: int = 521,
    niter_GPAreal: int = 3000,
    niter_VI: int = 100,
    tau_grid: Sequence[float] = (0.2, 0.4, 0.6, 0.8, 0.9),
) -> Dict[str, Any]:
    """
    Takes permuted data and runs:
      - GPArealModel (areal GP) on block averages
      - VIGP_Unlinked (VI for unlinked GP) on blockwise permuted X,Y,S

    This assumes permutation already done at the data level.
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    coords_perm = _to_tensor(coords_perm)
    Y_perm = _to_tensor(Y_perm).view(-1)  # (N,)
    X_perm = _to_tensor(X_perm)

    if X_perm.ndim == 1:
        X_perm = X_perm.unsqueeze(1)
    p = X_perm.shape[1]
    if p != 1:
        raise ValueError(f"Current VI wrapper assumes scalar X (p=1), but got p={p}.")

    region_assignments = region_assignments.to(device=device, dtype=torch.long)
    perm_matrix_x = perm_matrix_x.to(device=device)
    perm_matrix_s = perm_matrix_s.to(device=device)

    N = coords_perm.shape[0]
    assert N == n_blocks * n_locations, "N must equal n_blocks * n_locations."

    # ---------- Areal GP ----------
    ybar = torch.zeros(n_blocks, device=device)
    xbar = torch.zeros(n_blocks, p, device=device)

    for b in range(n_blocks):
        start = b * n_locations
        end = (b + 1) * n_locations
        idx = torch.arange(start, end, device=device)
        ybar[b] = Y_perm[idx].mean()
        xbar[b] = X_perm[idx].mean(dim=0)

    gpa = GPArealModel().to(device)
    opt_gpa = optim.AdamW(gpa.parameters(), lr=0.01, weight_decay=0.01)

    for _ in tqdm(range(niter_GPAreal), desc="Train GPArealModel (areal)"):
        opt_gpa.zero_grad()
        # jumbled locations + region assignments + block-averaged X,Y
        loss = gpa(coords_perm, region_assignments, xbar, ybar)
        loss.backward()
        opt_gpa.step()
        with torch.no_grad():
            gpa.sigmasq.clamp_(min=1e-6)
            gpa.phi.clamp_(min=1e-6)
            gpa.tausq.clamp_(min=1e-6)

    results: Dict[str, Any] = {}
    results["GPArealModel"] = {
        "nu": float(gpa.nu.item()),
        "phi": float(gpa.phi.item()),
        "sigmasq": float(gpa.sigmasq.item()),
        "tausq": float(gpa.tausq.item()),
        "beta": gpa.beta.detach().cpu().numpy(),
    }

    # ---------- VI for Unlinked GP ----------
    # Distances from permuted coordinates
    Dist = torch.cdist(coords_perm, coords_perm, p=2)
    Dist = (Dist + Dist.T) / 2.0

    # Block-shaped tensors for VI
    X_blocks = X_perm.view(n_blocks, n_locations)  # (B, n_i)
    Y_blocks = Y_perm.view(n_blocks, n_locations)  # (B, n_i)

    n_steps = 50
    n_phi_samples = 100
    n_piX_sample = 50
    n_piS_sample = 50

    prior_parameters = {
        "a1": 0.1,
        "b1": 0.1,
        "a2": 0.1,
        "b2": 0.1,
        "eta_X_sq": 0.1,
        "eta_S_sq": 0.1,
        "mu_beta": 0.0,
        "sigmasq_beta": 100.0,
        "phi_prior_lb": (1.0 / torch.max(Dist)),
        "phi_prior_ub": 10.0,
    }

    # Precompute mean_Rphi_inv_fixed at some phi (e.g., 4)
    Rphi = torch.exp(-4.0 * Dist)
    mean_Rphi_inv_fixed = torch.linalg.inv(Rphi + 1e-6 * torch.eye(N, device=device))

    for tau in tau_grid:
        vi_out = VIGP_Unlinked(
            n_iter=niter_VI,
            n_blocks=n_blocks,
            n_locations=n_locations,
            X=X_blocks,
            Y=Y_blocks,
            Dist=Dist,
            n_steps=n_steps,
            n_phi_samples=n_phi_samples,
            n_piX_sample=n_piX_sample,
            tau_X=tau,
            tau_S=tau,
            n_piS_sample=n_piS_sample,
            seed=seed,
            fix_piX=False,
            fix_piS=False,
            fix_mu_lambda_beta=False,
            fix_sigmasq_lambda_beta=False,
            fix_lambda_b1=False,
            lambda_b1_fixed=((n_blocks * n_locations) * 0.5 + 0.1) * 5,
            fix_lambda_b2=False,
            M_X_star_fixed=perm_matrix_x.T,
            M_S_star_fixed=perm_matrix_s.T,
            V_X_star_fixed=torch.eye(n_locations, n_locations, device=device),
            V_S_star_fixed=torch.eye(n_locations, n_locations, device=device),
            phi_init=0.5,
            mean_Rphi_inv_fixed=mean_Rphi_inv_fixed,
            fix_mean_Rphi_inv=False,
            pi_X_true=perm_matrix_x.T,
            pi_S_true=perm_matrix_s.T,
            VX_ub=0.5,
            VS_ub=0.5,
            lr_piS=0.01,
            lr_piX=0.01,
            prior_parameters=prior_parameters,
        )
        results[f"VIGP_unlinked_tau_{tau}"] = vi_out

    # Summary for quick comparison (add metrics later as you wish)
    summary = {
        "areal_GP": {
            "nu": results["GPArealModel"]["nu"],
            "phi": results["GPArealModel"]["phi"],
            "sigmasq": results["GPArealModel"]["sigmasq"],
            "tausq": results["GPArealModel"]["tausq"],
            "beta": results["GPArealModel"]["beta"],
        },
        "VI_unlinked": {
            "taus": list(tau_grid),
        },
    }

    results["summary"] = summary
    return results


# ============================================================
# Demo: FULL 5-STEP PIPELINE on meuse
# ============================================================

if ro is None:
    print("rpy2 not installed; skipping meuse demo. Install `rpy2` and have R + sp available.")
else:
    # -----------------------------
    # STEP 1: get data (meuse)
    # -----------------------------
    ro.r("suppressMessages(library(sp))")
    ro.r("data(meuse)")
    meuse_r = ro.r("meuse")
    meuse_df: pd.DataFrame = pandas2ri.rpy2py(meuse_r)

    coords_meuse = meuse_df[["x", "y"]].values
    Y_meuse = np.log1p(meuse_df["zinc"].values)  # log transform
    X_meuse = meuse_df["elev"].values           # scalar covariate

    N_meuse = coords_meuse.shape[0]
    print(f"Loaded meuse: N = {N_meuse}")

    # Choose block design
    n_blocks_demo = 25
    n_locations_demo = 6
    N_use_demo = n_blocks_demo * n_locations_demo
    if N_meuse < N_use_demo:
        raise ValueError(f"Meuse has {N_meuse} rows; reduce blocks or n_locations.")

    # -----------------------------
    # STEP 2: create permuted data
    # -----------------------------
    prep = make_blockwise_permuted_data(
        coords=coords_meuse,
        X=X_meuse,
        Y=Y_meuse,
        n_blocks=n_blocks_demo,
        n_locations=n_locations_demo,
        seed=2025,
    )

    coords_orig = prep["coords_orig"]
    X_orig = prep["X_orig"]
    Y_orig = prep["Y_orig"]

    coords_perm = prep["coords_perm"]
    X_perm = prep["X_perm"]
    Y_perm = prep["Y_perm"]
    region_assignments = prep["region_assignments"]
    perm_matrix_x = prep["perm_matrix_x"]
    perm_matrix_s = prep["perm_matrix_s"]

    # -----------------------------
    # STEP 3: Areal + VI on permuted data
    # -----------------------------
    out_areal_vi = run_areal_and_vi(
        coords_perm=coords_perm,
        X_perm=X_perm,
        Y_perm=Y_perm,
        region_assignments=region_assignments,
        perm_matrix_x=perm_matrix_x,
        perm_matrix_s=perm_matrix_s,
        n_blocks=n_blocks_demo,
        n_locations=n_locations_demo,
        seed=2025,
        niter_GPAreal=800,   # shorter for demo
        niter_VI=50,
        tau_grid=(0.3, 0.5, 0.7),
    )

    # -----------------------------
    # STEP 4: Oracle GP on original data
    # (run outside helper, as requested)
    # -----------------------------
    gp_oracle = GPModel().to(device)
    opt_gp = optim.AdamW(gp_oracle.parameters(), lr=0.01, weight_decay=0.01)

    for _ in tqdm(range(800), desc="Train GPModel (oracle, meuse)"):
        opt_gp.zero_grad()
        loss = gp_oracle(coords_orig, X_orig, Y_orig)
        loss.backward()
        opt_gp.step()
        with torch.no_grad():
            gp_oracle.sigmasq.clamp_(min=1e-6)
            gp_oracle.phi.clamp_(min=1e-6)
            gp_oracle.tausq.clamp_(min=1e-6)

    oracle_params = {
        "nu": float(gp_oracle.nu.item()),
        "phi": float(gp_oracle.phi.item()),
        "sigmasq": float(gp_oracle.sigmasq.item()),
        "tausq": float(gp_oracle.tausq.item()),
        "beta": gp_oracle.beta.detach().cpu().numpy(),
    }

    # -----------------------------
    # STEP 5: Compare results (simple print)
    # -----------------------------
    print("\n=== Oracle GP (original data) ===")
    print(oracle_params)

    print("\n=== Areal GP (permuted data) ===")
    print(out_areal_vi["summary"]["areal_GP"])

    print("\n=== VI (unlinked) taus tried ===")
    print(out_areal_vi["summary"]["VI_unlinked"]["taus"])
